# Demo of IPC Dynamic Simulation in 2D and 3D

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import sys; sys.path.append('..')
import MeshFEM
import mesh, elastic_solid, energy, sim_utils, loads, benchmark, py_newton_optimizer
import tri_mesh_viewer

In [ ]:
RECORD_VIDEO=False

In [ ]:
DIM = 2
DEG = 2
if DIM == 3:
    PATH = '../../misc/examples/meshes/bunny_coarse.msh'
    RHO = 1e-4 # mass density in kg/mm^3
    GROUND_TOL = 4 # Selection box height above ground to define pinned vertices
    T, dt = 10, 0.1
else:
    PATH = '../../misc/examples/meshes/square_hole_subdiv.msh'
    RHO = 4e-2
    GROUND_TOL = 1e-08
    T, dt = 2, 0.001

m = mesh.Mesh(PATH, degree=DEG)
es = elastic_solid.ElasticSolid(m, energy.NeoHookeanYoungPoisson(DIM, E=2, nu=0.4)) # Silicon material (Y= 2MPa, nu=0.4)

es.rho = RHO
g = loads.Gravity(es)

In [ ]:
v = tri_mesh_viewer.Viewer(es, wireframe=True)
v.makeOpaque(color='#48B3FF')
v.show()

In [ ]:
import meshfem_ipc

In [ ]:
contact = meshfem_ipc.IPCObjectiveTerm(es)

In [ ]:
import dynamic_simulator
ds = dynamic_simulator.DynamicSimulator(es, [g, contact], useLumpedMass=True, dt=dt)

In [ ]:
# Glue to the ground.
ds.fixedVars = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MIN_Y, tol=GROUND_TOL)

In [ ]:
contact.sparsityPatternUpdateThreshold = 10

In [ ]:
ds.method = ds.method.BackwardEuler
ds.setPostTimestepCallback(v.updater(updateFrequency=1))
ds.optimizer.options.verbose = 0

In [ ]:
benchmark.reset()
if RECORD_VIDEO: v.recordStart(f'ipc_{DIM}d_deg{DEG}.mp4', renderScale=4, outputScale=2, framerate=60)
cr = ds.run(T)
if RECORD_VIDEO: v.recordStop()
benchmark.report()